In [158]:
!pip install --no-index /kaggle/input/imc2024-packages-lightglue-rerun-kornia/* --no-deps
!pip install pycolmap
!pip install git+https://github.com/cvg/LightGlue.git

!apt-get install -y colmap

Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/kornia-0.7.2-py2.py3-none-any.whl
Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/kornia_moons-0.2.9-py3-none-any.whl
ERROR: kornia_rs-0.1.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl is not a supported wheel on this platform.
  Cloning https://github.com/cvg/LightGlue.git to /tmp/pip-req-build-n0b_yylg
  Running command git clone --filter=blob:none --quiet https://github.com/cvg/LightGlue.git /tmp/pip-req-build-n0b_yylg
  Resolved https://github.com/cvg/LightGlue.git to commit edb2b838efb2ecfe3f88097c5fad9887d95aedad
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
colmap is already the newest version (3.7-2).
0 upgraded, 0 newly installed, 0 to remove and 87 not upgraded.


In [159]:
import pandas as pd
import os
import sys
import shutil
import numpy as np
from scipy.spatial.transform import Rotation as R
import cv2
import torch
import pycolmap
import array
import h5py
import subprocess
import kornia as K
import kornia.feature as KF
from lightglue import ALIKED
from lightglue import LightGlue
sys.path.append('/kaggle/input/imc25-utils')
from database import COLMAPDatabase
from h5_to_db import add_keypoints, add_matches, import_into_colmap

In [160]:
data_path = "/kaggle/input/image-matching-challenge-2025"
test_path = os.path.join(data_path, "test")
train_path = os.path.join(data_path, "train")

# Загрузка sample_submission.csv
submission = pd.read_csv(os.path.join(data_path, "sample_submission.csv"))

In [161]:
def group_images_by_size(dataset, scene, group):
    print(f"Группировка для dataset={dataset}, scene={scene}, строк в группе={len(group)}")
    image_sizes = {}
    for idx, row in group.iterrows():
        print(f"Обрабатываем строку: {row.to_dict()}")
        # Пробуем сформировать путь
        image_path = os.path.join(dataset, row['image'])
        print(f"Путь к изображению: {image_path}")
        if os.path.exists(image_path):
            img = cv2.imread(image_path)
            if img is not None:
                size = img.shape[:2]
                size_key = f"{size[0]}x{size[1]}"
                print(f"Размер изображения: {size_key}")
                if size_key not in image_sizes:
                    image_sizes[size_key] = []
                image_sizes[size_key].append((idx, row))
            else:
                print(f"Не удалось прочитать изображение: {image_path}")
        else:
            print(f"Изображение не найдено: {image_path}")
    print(f"Итоговые группы: {[(k, len(v)) for k, v in image_sizes.items()]}")
    return image_sizes

In [162]:
def process_group(dataset, scene, group, indices):
    # Создание рабочей директории для проекта
    project_dir = f"/kaggle/working/project_{dataset}_{scene}_{indices[0]}"
    os.makedirs(project_dir, exist_ok=True)
    db_path = os.path.join(project_dir, "database.db")
    
    # Удаление старой базы данных, если она существует
    if os.path.exists(db_path):
        os.remove(db_path)
    
    # Копирование изображений в рабочую директорию
    image_names = []
    test_path = os.path.join(data_path, "test")  # Укажите правильный путь к данным
    for idx, row in group.iterrows():
        image_path = os.path.join(test_path, dataset, row['image'])
        if os.path.exists(image_path):
            dest_path = os.path.join(project_dir, os.path.basename(row['image']))
            shutil.copy(image_path, dest_path)
            image_names.append(os.path.basename(row['image']))
    
    # Пропуск группы, если изображений меньше 2
    if len(image_names) < 2:
        print(f"Skipping group with insufficient images: {len(image_names)}")
        shutil.rmtree(project_dir)
        return
    
    # Инициализация устройств и моделей
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    aliked = ALIKED(max_num_keypoints=4096).to(device)
    matcher = KF.LightGlueMatcher(
        "aliked",
        {
            "width_confidence": 0.7,
            "depth_confidence": 0.6,
            "mp": True if 'cuda' in str(device) else False
        }
    ).eval().to(device)
    
    # Создание директории для признаков
    feature_dir = os.path.join(project_dir, "features")
    os.makedirs(feature_dir, exist_ok=True)

    # Удаление существующих H5-файлов
    for h5_file in ['keypoints.h5', 'descriptors.h5', 'matches.h5']:
        h5_path = os.path.join(feature_dir, h5_file)
        if os.path.exists(h5_path):
            os.remove(h5_path)
            
    # Извлечение признаков с помощью ALIKED и сохранение в H5-файлы
    with h5py.File(os.path.join(feature_dir, 'keypoints.h5'), 'w') as f_kp, \
         h5py.File(os.path.join(feature_dir, 'descriptors.h5'), 'w') as f_desc:
        for image_name in image_names:
            img = cv2.imread(os.path.join(project_dir, image_name))
            if img is None:
                print(f"Ошибка чтения изображения: {image_name}")
                continue
            img_tensor = torch.from_numpy(img).permute(2, 0, 1).float().to(device) / 255.0
            feats = aliked.extract(img_tensor)
            kpts = feats['keypoints'].cpu().numpy()
            descs = feats['descriptors'].cpu().numpy()
            if kpts.ndim > 2:
                kpts = kpts.squeeze()  # Удаляем лишние размерности, например, [1, N, 2] -> [N, 2]
            if kpts.ndim == 1:
                kpts = kpts.reshape(-1, 2)  # На случай, если данные одномерные
            if descs.ndim == 3:  # Удаляем лишнюю размерность (1, N, 128) -> (N, 128)
                descs = descs.squeeze(0)

            print(f"Извлечено для {image_name}: {kpts.shape[0]} ключевых точек, дескрипторы: {descs.shape}")
            
            f_kp[image_name] = kpts
            f_desc[image_name] = descs
    keypoints = {}
    descriptors = {}
    with h5py.File(os.path.join(feature_dir, 'keypoints.h5'), 'r') as f_kp, \
         h5py.File(os.path.join(feature_dir, 'descriptors.h5'), 'r') as f_desc:
        for image_name in image_names:
            keypoints[image_name] = f_kp[image_name][...]
            descriptors[image_name] = f_desc[image_name][...]
            print(f"Загружено для {image_name}: ключевые точки {keypoints[image_name].shape}, дескрипторы {descriptors[image_name].shape}")
            
    # Сопоставление признаков с помощью LightGlue и сохранение в H5-файл
    with h5py.File(os.path.join(feature_dir, 'matches.h5'), 'w') as f_match:
        for i in range(len(image_names)):
            for j in range(i + 1, len(image_names)):
                image_name1, image_name2 = image_names[i], image_names[j]
                kp1 = keypoints[image_name1]
                kp2 = keypoints[image_name2]
                desc1 = descriptors[image_name1]
                desc2 = descriptors[image_name2]
                # Корректировка формы дескрипторов
                if desc1.ndim == 3:
                    desc1 = desc1.squeeze(0)
                if desc2.ndim == 3:
                    desc2 = desc2.squeeze(0)
                # Корректировка формы ключевых точек для LAF
                if kp1.ndim == 2:
                    kp1 = kp1[None, :]
                if kp2.ndim == 2:
                    kp2 = kp2[None, :]

                print(f"Перед сопоставлением {image_name1} vs {image_name2}:")
                print(f"  kp1: {kp1.shape}, kp2: {kp2.shape}")
                print(f"  desc1: {desc1.shape}, desc2: {desc2.shape}")
                
                # Преобразование в LAF
                laf1 = KF.laf_from_center_scale_ori(torch.from_numpy(kp1).to(device))
                laf2 = KF.laf_from_center_scale_ori(torch.from_numpy(kp2).to(device))
                print(f"  laf1: {laf1.shape}, laf2: {laf2.shape}")
                # Сопоставление
                with torch.inference_mode():
                    dists, idxs = matcher(
                        torch.from_numpy(desc1).to(device),
                        torch.from_numpy(desc2).to(device),
                        laf1,
                        laf2
                    )

                print(f"Совпадения между {image_name1} и {image_name2}: {len(idxs)}")
                
                # Сохранение совпадений
                if len(idxs) > 0:
                    matches = idxs.cpu().numpy().astype(np.uint32)
                    group = f_match.require_group(image_name1)
                    group.create_dataset(image_name2, data=matches.reshape(-1, 2))

    # Диагностика matches.h5
    print("\nСодержимое matches.h5:")
    with h5py.File(os.path.join(feature_dir, 'matches.h5'), 'r') as f_match:
        for g in f_match.keys():
            print(f"Группа: {g}, датасеты: {list(f_match[g].keys())}")
            for ds in f_match[g].keys():
                print(f"  {ds}: {f_match[g][ds][...].shape} совпадений")
    
    print("\nИмпорт данных в COLMAP...")
    import_into_colmap(
        img_dir=project_dir,
        feature_dir=feature_dir,
        database_path=db_path,
        img_ext='.png'
    )
    
    # Диагностика базы данных
    db = COLMAPDatabase.connect(db_path)
    print("\nСодержимое базы данных COLMAP:")
    cursor = db.execute("SELECT name FROM images")
    db_images = [row[0] for row in cursor.fetchall()]
    print(f"Изображения в базе: {db_images}")
    cursor = db.execute("SELECT SUM(rows) FROM keypoints")
    total_keypoints = cursor.fetchone()[0]
    print(f"Ключевые точки в базе (общее количество): {total_keypoints}")
    cursor = db.execute("SELECT image_id, rows FROM keypoints")
    keypoints_per_image = {db_images[i-1]: rows for i, rows in cursor.fetchall()}
    print(f"Ключевые точки по изображениям: {keypoints_per_image}")
    cursor = db.execute("SELECT pair_id, rows FROM matches WHERE rows > 0")
    matches_info = [(pair_id, rows) for pair_id, rows in cursor.fetchall()]
    print(f"Совпадения в базе (пары с ненулевыми совпадениями): {matches_info}")
    
    db.close()
    
    # Реконструкция сцены
    output_path = os.path.join(project_dir, "sparse")
    os.makedirs(output_path, exist_ok=True)
    mapper_options = pycolmap.IncrementalPipelineOptions()
    mapper_options.min_model_size = 2
    mapper_options.min_num_matches = 2
    mapper_options.multiple_models = True
    mapper_options.ba_global_max_num_iterations = 100
    try:
        maps = pycolmap.incremental_mapping(
            database_path=db_path,
            image_path=project_dir,
            output_path=output_path,
            options=mapper_options
        )
    except Exception as e:
        print(f"Ошибка при выполнении incremental_mapping: {e}")
        maps = {}
    
    # Извлечение кластеров
    clusters = []
    print(f"Результаты COLMAP: {len(maps)} моделей")
    reconstructed_images = set()
    if maps:
        print("maps check")
        for map_index, cur_map in maps.items():
            print(f"Модель {map_index}: {len(cur_map.images)} изображений")
            cluster = {
                'cluster_index': map_index,
                'images': []
            }
            for image_id, image in cur_map.images.items():
                image_name = image.name
                print(f"Изображение в модели: {image_name}")
                if image_name in image_names:
                    prediction_index = image_names.index(image_name)
                    row_idx = indices[prediction_index]
                    rotation = image.cam_from_world.rotation.matrix().flatten()
                    translation = image.cam_from_world.translation
                    cluster['images'].append({
                        'row_idx': row_idx,
                        'image_name': image_name,
                        'rotation': rotation,
                        'translation': translation
                    })
                    reconstructed_images.add(image_name)
            if cluster['images']:
                clusters.append(cluster)
    
    # Добавление выбросов
    for image_name in image_names:
        if image_name not in reconstructed_images:
            print(image_name, " is looser")
            prediction_index = image_names.index(image_name)
            row_idx = indices[prediction_index]
            clusters.append({
                'cluster_index': None,
                'images': [{
                    'row_idx': row_idx,
                    'image_name': image_name,
                    'rotation': None,
                    'translation': None
                }]
            })
    
    # Очистка
    shutil.rmtree(project_dir)
    
    return clusters

In [163]:
# Формирование submission.csv
def write_submission(clusters_list):
    submission_file = '/kaggle/working/submission.csv'
    array_to_str = lambda array: ';'.join([f"{x:.09f}" for x in array]) if array is not None else ';'.join(['nan'] * 9)
    none_to_str = lambda n: ';'.join(['nan'] * n)
    
    with open(submission_file, 'w') as f:
        f.write('image_id,dataset,scene,image,rotation_matrix,translation_vector\n')
        for clusters in clusters_list:
            for cluster in clusters:
                cluster_name = 'outliers' if cluster['cluster_index'] is None else f'cluster{cluster["cluster_index"]}'
                for img_data in cluster['images']:
                    row = submission.loc[img_data['row_idx']]
                    image_id = row['image_id']
                    dataset = row['dataset']
                    image_path = row['image']
                    rotation = array_to_str(img_data['rotation'])
                    translation = none_to_str(3) if img_data['translation'] is None else ';'.join([f"{x:.09f}" for x in img_data['translation']])
                    f.write(f'{image_id},{dataset},{cluster_name},{image_path},{rotation},{translation}\n')
    
    print(f"Submission file created: {submission_file}")
    with open(submission_file, 'r') as f:
        print("\nПервые строки submission.csv:")
        for _ in range(5):
            print(f.readline().strip())

In [164]:
# Основной цикл
all_clusters = []
groups = submission.groupby(['dataset', 'scene'])

# Установка headless-режима для COLMAP
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

for (dataset, scene), group in groups:
    dataset_path = os.path.join(test_path, dataset)
    print(dataset_path)
    if not os.path.exists(dataset_path):
        print(f"Пропуск {dataset}")
        continue
    
    image_groups = group_images_by_size(dataset_path, scene, group)
    for size_key, size_group in image_groups.items():
        print(f"Обработка {size_key}")
        temp_group = pd.DataFrame([row for _, row in size_group])
        temp_indices = [idx for idx, _ in size_group]
        clusters = process_group(dataset, scene, temp_group, temp_indices)
        if clusters:
            all_clusters.append(clusters)

# Запись результатов
write_submission(all_clusters)

/kaggle/input/image-matching-challenge-2025/test/ETs
Группировка для dataset=/kaggle/input/image-matching-challenge-2025/test/ETs, scene=cluster0, строк в группе=22
Обрабатываем строку: {'image_id': 'ETs_another_et_another_et001.png_public', 'dataset': 'ETs', 'scene': 'cluster0', 'image': 'another_et_another_et001.png', 'rotation_matrix': '0.984538905;0.811159853;0.681088421;0.845092853;0.638842711;0.183570696;0.075868286;0.663749411;0.658267773', 'translation_vector': '0.427962849;0.408638396;0.184856919'}
Путь к изображению: /kaggle/input/image-matching-challenge-2025/test/ETs/another_et_another_et001.png
Размер изображения: 640x360
Обрабатываем строку: {'image_id': 'ETs_another_et_another_et002.png_public', 'dataset': 'ETs', 'scene': 'cluster0', 'image': 'another_et_another_et002.png', 'rotation_matrix': '0.557144821;0.589081187;0.788143210;0.101204140;0.708863497;0.435696397;0.539207526;0.903524615;0.308633579', 'translation_vector': '0.112591455;0.280966383;0.338320921'}
Путь к из

100%|██████████| 10/10 [00:00<00:00, 118.08it/s]
45it [00:00, 4470.69it/s]             


Содержимое базы данных COLMAP:
Изображения в базе: ['another_et_another_et001.png', 'another_et_another_et002.png', 'another_et_another_et003.png', 'another_et_another_et004.png', 'another_et_another_et005.png', 'another_et_another_et006.png', 'another_et_another_et007.png', 'another_et_another_et008.png', 'another_et_another_et009.png', 'another_et_another_et010.png']
Ключевые точки в базе (общее количество): 13408
Ключевые точки по изображениям: {'another_et_another_et001.png': 1548, 'another_et_another_et002.png': 1387, 'another_et_another_et003.png': 1404, 'another_et_another_et004.png': 1389, 'another_et_another_et005.png': 1352, 'another_et_another_et006.png': 1305, 'another_et_another_et007.png': 1410, 'another_et_another_et008.png': 1411, 'another_et_another_et009.png': 1163, 'another_et_another_et010.png': 1039}
Совпадения в базе (пары с ненулевыми совпадениями): [(2147483649, 889), (2147483650, 368), (2147483651, 604), (2147483652, 687), (2147483653, 329), (2147483654, 188),


I20250525 19:09:33.348646 135477238813824 incremental_pipeline.cc:237] Loading database
I20250525 19:09:33.350285 135477238813824 database_cache.cc:66] Loading cameras...
I20250525 19:09:33.350359 135477238813824 database_cache.cc:76]  10 in 0.000s
I20250525 19:09:33.350380 135477238813824 database_cache.cc:84] Loading matches...
I20250525 19:09:33.350400 135477238813824 database_cache.cc:89]  0 in 0.000s
I20250525 19:09:33.350411 135477238813824 database_cache.cc:105] Loading images...
I20250525 19:09:33.350486 135477238813824 database_cache.cc:153]  10 in 0.000s (connected 0)
I20250525 19:09:33.350505 135477238813824 database_cache.cc:164] Loading pose priors...
I20250525 19:09:33.350536 135477238813824 database_cache.cc:175]  0 in 0.000s
I20250525 19:09:33.350551 135477238813824 database_cache.cc:184] Building correspondence graph...
I20250525 19:09:33.350558 135477238813824 database_cache.cc:210]  in 0.000s (ignored 0)
I20250525 19:09:33.350569 135477238813824 timer.cc:91] Elapsed

Loaded LightGlue model
Извлечено для et_et000.png: 2587 ключевых точек, дескрипторы: (2587, 128)
Извлечено для et_et001.png: 2595 ключевых точек, дескрипторы: (2595, 128)
Извлечено для et_et002.png: 2375 ключевых точек, дескрипторы: (2375, 128)
Извлечено для et_et003.png: 2120 ключевых точек, дескрипторы: (2120, 128)
Извлечено для et_et004.png: 2541 ключевых точек, дескрипторы: (2541, 128)
Извлечено для et_et005.png: 2235 ключевых точек, дескрипторы: (2235, 128)
Извлечено для et_et006.png: 1998 ключевых точек, дескрипторы: (1998, 128)
Извлечено для et_et007.png: 2006 ключевых точек, дескрипторы: (2006, 128)
Извлечено для et_et008.png: 2400 ключевых точек, дескрипторы: (2400, 128)
Загружено для et_et000.png: ключевые точки (2587, 2), дескрипторы (2587, 128)
Загружено для et_et001.png: ключевые точки (2595, 2), дескрипторы (2595, 128)
Загружено для et_et002.png: ключевые точки (2375, 2), дескрипторы (2375, 128)
Загружено для et_et003.png: ключевые точки (2120, 2), дескрипторы (2120, 128)

100%|██████████| 9/9 [00:00<00:00, 81.15it/s]
36it [00:00, 4453.21it/s]             
I20250525 19:09:34.749655 135477238813824 incremental_pipeline.cc:237] Loading database
I20250525 19:09:34.751155 135477238813824 database_cache.cc:66] Loading cameras...
I20250525 19:09:34.751205 135477238813824 database_cache.cc:76]  9 in 0.000s
I20250525 19:09:34.751217 135477238813824 database_cache.cc:84] Loading matches...
I20250525 19:09:34.751228 135477238813824 database_cache.cc:89]  0 in 0.000s
I20250525 19:09:34.751233 135477238813824 database_cache.cc:105] Loading images...
I20250525 19:09:34.751290 135477238813824 database_cache.cc:153]  9 in 0.000s (connected 0)
I20250525 19:09:34.751304 135477238813824 database_cache.cc:164] Loading pose priors...
I20250525 19:09:34.751331 135477238813824 database_cache.cc:175]  0 in 0.000s
I20250525 19:09:34.751341 135477238813824 database_cache.cc:184] Building correspondence graph...
I20250525 19:09:34.751348 135477238813824 database_cache.cc:210]  in


Содержимое базы данных COLMAP:
Изображения в базе: ['et_et000.png', 'et_et001.png', 'et_et002.png', 'et_et003.png', 'et_et004.png', 'et_et005.png', 'et_et006.png', 'et_et007.png', 'et_et008.png']
Ключевые точки в базе (общее количество): 20857
Ключевые точки по изображениям: {'et_et000.png': 2587, 'et_et001.png': 2595, 'et_et002.png': 2375, 'et_et003.png': 2120, 'et_et004.png': 2541, 'et_et005.png': 2235, 'et_et006.png': 1998, 'et_et007.png': 2006, 'et_et008.png': 2400}
Совпадения в базе (пары с ненулевыми совпадениями): [(2147483649, 1191), (2147483650, 734), (2147483651, 1537), (2147483652, 800), (2147483653, 84), (2147483654, 74), (2147483655, 80), (2147483656, 59), (4294967297, 1248), (4294967298, 866), (4294967299, 889), (4294967300, 85), (4294967301, 78), (4294967302, 87), (4294967303, 74), (6442450945, 535), (6442450946, 722), (6442450947, 150), (6442450948, 188), (6442450949, 122), (6442450950, 81), (8589934593, 586), (8589934594, 62), (8589934595, 58), (8589934596, 73), (8589

100%|██████████| 51/51 [00:02<00:00, 23.28it/s]
1275it [00:00, 5501.42it/s]                          


Содержимое базы данных COLMAP:
Изображения в базе: ['stairs_split_1_1710453576271.png', 'stairs_split_1_1710453601885.png', 'stairs_split_1_1710453606287.png', 'stairs_split_1_1710453612890.png', 'stairs_split_1_1710453616892.png', 'stairs_split_1_1710453620694.png', 'stairs_split_1_1710453626698.png', 'stairs_split_1_1710453643106.png', 'stairs_split_1_1710453651110.png', 'stairs_split_1_1710453659313.png', 'stairs_split_1_1710453663515.png', 'stairs_split_1_1710453667117.png', 'stairs_split_1_1710453668718.png', 'stairs_split_1_1710453675921.png', 'stairs_split_1_1710453678922.png', 'stairs_split_1_1710453683725.png', 'stairs_split_1_1710453689727.png', 'stairs_split_1_1710453693529.png', 'stairs_split_1_1710453697531.png', 'stairs_split_1_1710453704934.png', 'stairs_split_1_1710453901046.png', 'stairs_split_1_1710453912451.png', 'stairs_split_1_1710453930259.png', 'stairs_split_1_1710453947066.png', 'stairs_split_1_1710453955270.png', 'stairs_split_1_1710453963274.png', 'stairs_spl


I20250525 19:10:04.301416 135477238813824 incremental_pipeline.cc:237] Loading database
I20250525 19:10:04.302968 135477238813824 database_cache.cc:66] Loading cameras...
I20250525 19:10:04.303035 135477238813824 database_cache.cc:76]  51 in 0.000s
I20250525 19:10:04.303046 135477238813824 database_cache.cc:84] Loading matches...
I20250525 19:10:04.303058 135477238813824 database_cache.cc:89]  0 in 0.000s
I20250525 19:10:04.303062 135477238813824 database_cache.cc:105] Loading images...
I20250525 19:10:04.303145 135477238813824 database_cache.cc:153]  51 in 0.000s (connected 0)
I20250525 19:10:04.303165 135477238813824 database_cache.cc:164] Loading pose priors...
I20250525 19:10:04.303207 135477238813824 database_cache.cc:175]  0 in 0.000s
I20250525 19:10:04.303217 135477238813824 database_cache.cc:184] Building correspondence graph...
I20250525 19:10:04.303224 135477238813824 database_cache.cc:210]  in 0.000s (ignored 0)
I20250525 19:10:04.303233 135477238813824 timer.cc:91] Elapsed